# lm-eval on vLLM with transformers 5: a compatibility check with Tiny Aya

This notebook checks that the **transformers 5 + vLLM** stack AuditKIT now targets (`vllm>=0.30`, `transformers>=5.15,<6`) works end to end on a Cohere model. It uses **Tiny Aya Global** (`cohere2`).

1. **Environment**: `check_compat()` confirms the resolved versions: vLLM 0.30+, transformers 5.x, and torch pinned by vLLM.
2. **Native vLLM backend**: `ak.evaluate(..., model=VLLMModel(...))` loads the model in vLLM and generates with its chat template.
3. **lm-eval through vLLM**: `ak.run_benchmark([...], "vllm:CohereLabs/tiny-aya-global")` runs a loglikelihood task (`arc_easy`) and a generative task (`gsm8k`, with the chat template).
4. **Parity** (optional): the same tasks on the `hf:` backend. The scores should be close.
5. A **pass/fail checklist**.

**Runtime:** a Colab **L4 or A100**. vLLM's current wheels target Ampere or newer; a T4 may not be supported by the installed vLLM build. vLLM needs Linux + CUDA, so it doesn't run on macOS.

**Access:** Tiny Aya is gated. Accept the licence and add the `HF_TOKEN` secret, plus `GITHUB_TOKEN` to install AuditKIT.

In [ ]:
# ---- settings ----
import os
MODEL = os.environ.get("NB_MODEL", "CohereLabs/tiny-aya-global")
TASKS = ["arc_easy", "gsm8k"]      # loglikelihood + generative
LIMIT = int(os.environ.get("NB_LIMIT", "50"))   # samples per task (raise for a real score)
RUN_HF_PARITY = True               # also run the tasks on hf: for comparison (slower)
VLLM_ARGS = {"gpu_memory_utilization": 0.85, "max_model_len": 4096}

In [ ]:
# ---- install (Colab) / reuse (local checkout) ----
# Colab: clones AuditKIT at REPO_BRANCH and installs it with the extras this notebook needs.
# Local: uses the auditkit already importable (e.g. the repo's .venv kernel).
# Installing [vllm] brings vLLM >= 0.30 and the torch/transformers 5 versions it pins; do it before importing torch.
import os, sys, subprocess
REPO_BRANCH = "feat/rag-agent-evals"   # #1's branch: #9, #10 and every #15 / #27 fix are merged here
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    for key in ("GITHUB_TOKEN", "HF_TOKEN"):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
    repo_url = f"https://{os.environ['GITHUB_TOKEN']}@github.com/aryapratinavseth/AuditKIT-Internal.git"
    if not os.path.isdir("/content/AuditKIT-Internal"):
        subprocess.run(["git", "clone", "-q", "--branch", REPO_BRANCH, repo_url, "/content/AuditKIT-Internal"], check=True)
    else:   # a reused runtime: move the existing clone to the branch head, never run stale code
        subprocess.run(["git", "-C", "/content/AuditKIT-Internal", "fetch", "-q", repo_url, REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", "/content/AuditKIT-Internal", "checkout", "-q", "-B", REPO_BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e",
                    "/content/AuditKIT-Internal[vllm,lmeval]"], check=True)
    # vLLM >= 0.30 upgrades torch to a CUDA 13 build. Colab's preinstalled torchaudio (CUDA 12.8) is not a vLLM
    # dependency, so pip leaves it behind, and transformers imports it if present -> "PyTorch and TorchAudio were
    # compiled with different CUDA versions". Nothing here uses audio, so remove it before anything imports torch.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"], check=False)
    print(subprocess.run([sys.executable, "-c", "import torch, torchvision; print('torch', torch.__version__, "
                          "'cuda', torch.version.cuda, '| torchvision', torchvision.__version__)"],
                         capture_output=True, text=True).stdout)
    sys.path.insert(0, "/content/AuditKIT-Internal/src")

if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
else:
    print("HF_TOKEN not set: gated Cohere models (Tiny Aya, Aya Expanse, Aya Vision) will fail to download.")

In [ ]:
from auditkit.compat import check_compat
report = check_compat(check_cuda=True)
print(report)
v = report.versions
print("\ntransformers", v.get("transformers"), "| vllm", v.get("vllm"), "| torch", v.get("torch"))

In [ ]:
import auditkit as ak, torch, transformers
from auditkit.runspec import RunConfig
from auditkit.model.hf_gen import HFGenModel
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
DTYPE = torch.bfloat16 if (BF16 or DEVICE == "mps") else torch.float16
print(f"auditkit {ak.__version__} | transformers {transformers.__version__} | torch {torch.__version__}")
print(f"device {DEVICE} | GPU {GPU_GB:.0f} GB | dtype {DTYPE}")
assert int(transformers.__version__.split(".")[0]) >= 5, "the Cohere models need transformers >= 5.15"

## 1 · Native vLLM backend

The `vllm:` backend builds a `vllm.LLM`, renders each prompt with the model's chat template, and generates. If the tokenizer, template or architecture were not compatible, it would fail here.

In [ ]:
from auditkit import Sample
from auditkit.model.vllm_gen import VLLMModel
QA = [("What is the capital of France? Answer with one word.", "Paris"),
      ("Translate to Spanish, reply with the translation only: thank you", "gracias"),
      ("What is 12 times 12? Answer with the number only.", "144"),
      ("Quelle est la capitale du Japon ? Réponds en un mot.", "Tokyo"),
      ("How many days are in a leap year? Answer with the number only.", "366")]
qa = [Sample(input=q, target=a) for q, a in QA]
# kwargs go straight to vllm.LLM(); the gated download uses the HF login above
llm = VLLMModel(MODEL, dtype="bfloat16" if BF16 else "float16", **VLLM_ARGS)
native = ak.evaluate(qa, model=llm, scorers=["quasi_exact_match", "f1_score"], config=RunConfig(max_tokens=32))
print("headline:", native.headline, "| errors:", native.errors)
for p in native.predictions:
    print(f"  {p.expected!r:>10} <- {(p.raw_output or '').strip()[:60]!r}")
llm.unload()        # free the GPU before lm-eval builds its own engine

## 2 · lm-eval through vLLM

`run_benchmark` maps `vllm:<model>` to lm-eval's `vllm` backend. `model_args` go to the engine, and `apply_chat_template=True` because Tiny Aya is an instruct model. It returns an AuditKIT `RunResult`, the same object `evaluate()` returns.

In [ ]:
bench_vllm = ak.run_benchmark(
    TASKS, f"vllm:{MODEL}",
    config=RunConfig(limit=LIMIT, num_fewshot=0, seed=0),
    apply_chat_template=True,
    gen_kwargs="temperature=0,max_gen_toks=256",
    model_args={**VLLM_ARGS, "dtype": "bfloat16" if BF16 else "float16"},
    **({"hf_token": os.environ["HF_TOKEN"]} if os.environ.get("HF_TOKEN") else {}),
)
print("vLLM headline:", {k: round(v, 3) for k, v in bench_vllm.headline.items()})
print("errors:", bench_vllm.errors[:3])

## 3 · Parity: the same tasks on `hf:` (optional)

In [ ]:
bench_hf = None
if RUN_HF_PARITY:
    import gc; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
    bench_hf = ak.run_benchmark(
        TASKS, f"hf:{MODEL}",
        config=RunConfig(limit=LIMIT, num_fewshot=0, seed=0, batch_size=8),
        apply_chat_template=True, gen_kwargs="temperature=0,max_gen_toks=256",
        model_args={"dtype": "bfloat16" if BF16 else "float16"},
        **({"hf_token": os.environ["HF_TOKEN"]} if os.environ.get("HF_TOKEN") else {}),
    )
    print(f"{'metric':32s} {'vllm':>8s} {'hf':>8s}")
    for k in sorted(set(bench_vllm.headline) | set(bench_hf.headline)):
        a, b = bench_vllm.headline.get(k), bench_hf.headline.get(k)
        print(f"{k:32s} {a if a is None else round(a, 3)!s:>8s} {b if b is None else round(b, 3)!s:>8s}")

## 4 · Checklist

In [ ]:
def major_minor(s):
    return tuple(int(x) for x in (s or "0.0").split(".")[:2])

checks = {
    "transformers >= 5.15": major_minor(v.get("transformers")) >= (5, 15),
    "vllm >= 0.30": major_minor(v.get("vllm")) >= (0, 30),
    "no error-level compat findings": report.ok,
    "native vLLM generation ran without errors": not native.errors,
    "lm-eval via vLLM produced scores": bool(bench_vllm.headline),
    "hf vs vLLM within 0.1 on every metric": (bench_hf is None) or all(
        abs(bench_vllm.headline[k] - bench_hf.headline[k]) <= 0.1
        for k in bench_vllm.headline if k in bench_hf.headline),
}
for name, ok in checks.items():
    print(("PASS " if ok else "FAIL ") + name)